# 🤖 Demand Forecasting — Model Training & Comparison

This notebook trains our core demand forecasting model and evaluates its performance.

In [ ]:
import sys
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

from src.utils.config import *
from src.data.data_pipeline import DataPipeline
from src.data.feature_engineering import FeatureEngineer
from src.models.demand_forecaster import DemandForecaster
from src.models.model_evaluator import ModelEvaluator
from src.visualization.plots import plot_actual_vs_predicted, plot_feature_importance

## 1. Data Preparation
We run the full pipeline to get the preprocessed train, val, and test splits.

In [ ]:
data_path = RAW_DATA_DIR / 'sales_data.csv'
df_raw = pd.read_csv(data_path)

pipeline = DataPipeline()
df_clean = pipeline.clean_data(df_raw)

engineer = FeatureEngineer()
df_features = engineer.create_features(df_clean)

train_df, val_df, test_df = pipeline.split_data(df_features)

## 2. Encode Categoricals
LightGBM natively handles categorical features if they are of 'category' dtype.

In [ ]:
X_train = train_df[ALL_FEATURES].copy()
y_train = train_df[TARGET_COLUMN].copy()

X_val = val_df[ALL_FEATURES].copy()
y_val = val_df[TARGET_COLUMN].copy()

X_test = test_df[ALL_FEATURES].copy()
y_test = test_df[TARGET_COLUMN].copy()

for col in CATEGORICAL_FEATURES:
    if col in X_train.columns:
        X_train[col] = X_train[col].astype('category')
        X_val[col] = X_val[col].astype('category')
        X_test[col] = X_test[col].astype('category')

## 3. Train LightGBM
We instantiate our wrapper and train the LGBMRegressor.

In [ ]:
model = DemandForecaster()
model.train(X_train, y_train, X_val, y_val)
print("Model training completed.")

## 4. Evaluate on Test Set
Let's see how our model performs on unseen test data.

In [ ]:
y_pred = model.predict(X_test)
evaluator = ModelEvaluator()
metrics = evaluator.evaluate("LightGBM", y_test, y_pred)
display(pd.DataFrame([metrics]))

## 5. Actual vs Predicted
A visual comparison over a subset of the test data.

In [ ]:
# Plot for first 200 samples for clarity
fig = plot_actual_vs_predicted(y_test.values[:200], y_pred[:200], title="Actual vs Predicted Demand (Test Set Sample)")
fig.show()

## 6. Residual Analysis
Analyzing residuals to ensure there is no structured error.

In [ ]:
residuals = y_test - y_pred

fig1 = px.histogram(residuals, title="Distribution of Residuals")
fig1.show()

fig2 = px.scatter(x=y_pred, y=residuals, labels={'x': 'Predicted Demand', 'y': 'Residuals'}, title="Residuals vs Predicted")
fig2.add_hline(y=0, line_dash="dash", line_color="red")
fig2.show()

## 7. Feature Importance
Which features were most useful for the model?

In [ ]:
importance_df = model.get_feature_importance()
fig = plot_feature_importance(importance_df, top_n=15, title="Top 15 Feature Importances")
fig.show()

## 8. LightGBM Parameters
These are the hyperparameters used from the config.

In [ ]:
print("LightGBM Hyperparameters:")
for key, value in LGBM_PARAMS.items():
    print(f"{key}: {value}")

## 9. Save Model
Persist the model for the optimization module.

In [ ]:
model_path = MODELS_DIR / 'lgbm_demand_model.pkl'
model.save(model_path)
print(f"Model saved to {model_path}")

## 10. Summary

- The LightGBM model successfully captures demand patterns.
- Evaluated metrics show the model's accuracy on unseen test data.
- Feature importance analysis highlights the key drivers of demand (often price, competitor price, and lag features).